# Notebook 2 — VGG-19 Classification Using XML Bounding-Box Crops

This notebook uses:

- `splits_top15(2).json` for the official top-15 train/validation/test membership and remapped labels `0–14`;
- original XML annotation files for the bounding-box coordinates;
- detection images as the source images.

It deliberately does **not** use `boxes_top15(2).json`.

Pipeline:

```text
split JSON chooses image and remapped class
    → matching XML supplies one or more object boxes
    → crop every valid object box
    → resize crop
    → VGG-19 predicts one of 15 classes
```

A single detection image can create multiple crop samples when its XML contains multiple `<object>` elements.

In [7]:
# 1. Imports

import json
import random
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

## 2. Configuration

Update `DETECTION_IMAGE_DIR` and `XML_DIR` to match your extracted detection dataset.

The parser expects Pascal VOC-style XML containing tags such as:

```xml
<object>
    <name>...</name>
    <bndbox>
        <xmin>...</xmin>
        <ymin>...</ymin>
        <xmax>...</xmax>
        <ymax>...</ymax>
    </bndbox>
</object>
```

In [ ]:
# 2. Configuration

SEED = 42

WORKSPACE_ROOT = Path.cwd()
NOTEBOOK_ROOT = next(
    (candidate for candidate in (
        WORKSPACE_ROOT,
        WORKSPACE_ROOT / "FarmPestManagementAI",
    ) if (candidate / "splits_top15.json").exists()),
    WORKSPACE_ROOT,
 )
DATA_PARENT = NOTEBOOK_ROOT.parent / "pest-management-dataset"

# Change these two paths to the actual detection image and XML folders.
DETECTION_IMAGE_DIR = (
    DATA_PARENT
    / "IP102_v1.1-20260731T031124Z-1-001"
    / "IP102_v1.1"
    / "Detection"
    / "VOC2007"
    / "JPEGImages"
    / "JPEGImages"
)
XML_DIR = (
    DATA_PARENT
    / "IP102_v1.1-20260731T031124Z-1-001"
    / "IP102_v1.1"
    / "Detection"
    / "VOC2007"
    / "Annotations"
    / "Annotations"
)

SPLITS_JSON = NOTEBOOK_ROOT / "splits_top15.json"

IMAGE_SIZE = 128
NUM_CLASSES = 15
NUM_EPOCHS = 10
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 5e-4

# Number of JSON image records to inspect from each split.
# None means all records.
MAX_TRAIN_IMAGES = None
MAX_VAL_IMAGES = None
MAX_TEST_IMAGES = None

# Each XML may contain multiple object boxes.
# None means keep every valid object crop.
MAX_CROPS_PER_IMAGE = None

BOX_PADDING = 0.05
MIN_CROP_WIDTH = 4
MIN_CROP_HEIGHT = 4

STRICT_VGG19_CLASSIFIER = True
SMALL_CLASSIFIER_UNITS = 512

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()
BEST_MODEL_PATH = Path("xml_crop_vgg19_best.pth")
CROP_MANIFEST_PATH = Path("xml_crop_manifest.json")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", device)

Device: cpu


## 3. Load the top-15 split JSON

Each split record has:

```python
["filename.jpg", remapped_label]
```

The JSON determines which source image belongs to training, validation, or testing. This preserves a strict image-level separation: crops from one source image cannot appear in another split.

In [9]:
# 3. Load and validate the split JSON

required_paths = [
    DETECTION_IMAGE_DIR,
    XML_DIR,
    SPLITS_JSON,
]

missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    raise FileNotFoundError(
        "The following required paths do not exist:\n"
        + "\n".join(str(path) for path in missing_paths)
        + "\nUpdate the configuration paths before continuing."
    )

with SPLITS_JSON.open("r", encoding="utf-8") as file:
    split_data = json.load(file)

required_splits = {"train", "val", "test"}
missing_splits = required_splits - set(split_data)

if missing_splits:
    raise ValueError(
        f"Split JSON is missing: {sorted(missing_splits)}"
    )

available_labels = sorted({
    int(label)
    for split_name in required_splits
    for _, label in split_data[split_name]
})

if available_labels != list(range(NUM_CLASSES)):
    raise ValueError(
        f"Expected labels 0-{NUM_CLASSES - 1}, "
        f"found {available_labels}."
    )

for split_name in ("train", "val", "test"):
    print(
        f"{split_name}: "
        f"{len(split_data[split_name]):,} source images"
    )

train: 6,748 source images
val: 1,446 source images
test: 1,447 source images


## 4. Select an optional class-balanced number of source images

Limiting occurs at the **source-image level**, before XML boxes are expanded into crops. This keeps the split membership clear and prevents accidentally sampling individual crops from the same image into different sets.

In [ ]:
# 2. Configuration



SEED = 42



NOTEBOOK_ROOT = Path(__file__).resolve().parents[1]
PROJECT_ROOT = NOTEBOOK_ROOT.parent
DATA_PARENT = PROJECT_ROOT / "pest-management-dataset"



# Change these two paths to the actual detection image and XML folders.

DETECTION_IMAGE_DIR = (

    DATA_PARENT

    / "IP102_v1.1-20260731T031124Z-1-001"

    / "IP102_v1.1"

    / "Detection"

    / "VOC2007"

    / "JPEGImages"

    / "JPEGImages"

)

XML_DIR = (

    DATA_PARENT

    / "IP102_v1.1-20260731T031124Z-1-001"

    / "IP102_v1.1"

    / "Detection"

    / "VOC2007"

    / "Annotations"

    / "Annotations"

)



SPLITS_JSON = NOTEBOOK_ROOT / "splits_top15.json"



IMAGE_SIZE = 128

NUM_CLASSES = 15

NUM_EPOCHS = 5

BATCH_SIZE = 8

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 5e-4



# Number of JSON image records to inspect from each split.

# None means all records.
# Smaller defaults keep the notebook fast for smoke runs.

MAX_TRAIN_IMAGES = 120

MAX_VAL_IMAGES = 30

MAX_TEST_IMAGES = 30



# Each XML may contain multiple object boxes.

# None means keep every valid object crop.

MAX_CROPS_PER_IMAGE = None



BOX_PADDING = 0.05

MIN_CROP_WIDTH = 4

MIN_CROP_HEIGHT = 4



STRICT_VGG19_CLASSIFIER = True

SMALL_CLASSIFIER_UNITS = 512



NUM_WORKERS = 0

PIN_MEMORY = torch.cuda.is_available()

BEST_MODEL_PATH = Path("xml_crop_vgg19_best.pth")

CROP_MANIFEST_PATH = Path("xml_crop_manifest.json")



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)



print("Device:", device)

## 5. Parse Pascal VOC XML into crop records

Each crop record stores:

```text
source filename
XML filename
object index
XML object name
[left, top, right, bottom]
remapped top-15 class label
```

The remapped training target comes from `splits_top15.json`. The XML object name is retained for diagnostics.

Because the provided files here do not include the original XML directory or a top-15 name-remapping table, the notebook cannot prove that every XML `<name>` matches the JSON remapped label. It reports images containing multiple distinct XML names so you can inspect them.

In [ ]:
# 5. XML parser and crop-manifest builder

def xml_path_for_image(xml_dir, filename):
    return Path(xml_dir) / f"{Path(filename).stem}.xml"


def parse_voc_objects(xml_path):
    try:
        xml_text = Path(xml_path).read_text(encoding="utf-8").lstrip()

        if xml_text.count("<annotation") > 1:
            closing_tag = "</annotation>"
            first_closing = xml_text.find(closing_tag)

            if first_closing != -1:
                xml_text = xml_text[: first_closing + len(closing_tag)]

        root = ET.fromstring(xml_text)
    except ET.ParseError as error:
        raise ValueError(f"Invalid XML file: {xml_path}") from error

    objects = []

    for object_index, object_node in enumerate(root.findall("object")):
        name_node = object_node.find("name")
        box_node = object_node.find("bndbox")

        if box_node is None:
            continue

        coordinate_nodes = {
            tag: box_node.find(tag)
            for tag in ("xmin", "ymin", "xmax", "ymax")
        }

        if any(node is None for node in coordinate_nodes.values()):
            continue

        try:
            left = float(coordinate_nodes["xmin"].text)
            top = float(coordinate_nodes["ymin"].text)
            right = float(coordinate_nodes["xmax"].text)
            bottom = float(coordinate_nodes["ymax"].text)
        except (TypeError, ValueError):
            continue

        object_name = (
            name_node.text.strip()
            if name_node is not None and name_node.text
            else "unknown"
        )

        objects.append({
            "object_index": object_index,
            "object_name": object_name,
            "box": [left, top, right, bottom],
        })

    return objects


def build_crop_records(
    image_records,
    image_dir,
    xml_dir,
    max_crops_per_image=None,
):
    crop_records = []
    missing_images = []
    missing_xml = []
    invalid_xml = []
    empty_xml = []
    multi_name_images = []

    for filename, remapped_label in tqdm(
        image_records,
        desc="Building crop manifest",
        leave=False,
    ):
        image_path = Path(image_dir) / filename
        xml_path = xml_path_for_image(xml_dir, filename)

        if not image_path.exists():
            missing_images.append(filename)
            continue

        if not xml_path.exists():
            missing_xml.append(filename)
            continue

        try:
            objects = parse_voc_objects(xml_path)
        except ValueError as error:
            invalid_xml.append({
                "filename": filename,
                "xml_filename": xml_path.name,
                "error": str(error),
            })
            continue

        if not objects:
            empty_xml.append(filename)
            continue

        distinct_names = sorted({
            item["object_name"]
            for item in objects
        })

        if len(distinct_names) > 1:
            multi_name_images.append({
                "filename": filename,
                "xml_names": distinct_names,
                "json_label": int(remapped_label),
            })

        if max_crops_per_image is not None:
            objects = objects[:max_crops_per_image]

        for item in objects:
            crop_records.append({
                "filename": filename,
                "xml_filename": xml_path.name,
                "object_index": item["object_index"],
                "object_name": item["object_name"],
                "box": item["box"],
                "label": int(remapped_label),
            })

    diagnostics = {
        "missing_images": missing_images,
        "missing_xml": missing_xml,
        "invalid_xml": invalid_xml,
        "empty_xml": empty_xml,
        "multi_name_images": multi_name_images,
    }

    return crop_records, diagnostics


crop_records_by_split = {}
diagnostics_by_split = {}

for split_name in ("train", "val", "test"):
    records, diagnostics = build_crop_records(
        limited_image_records[split_name],
        DETECTION_IMAGE_DIR,
        XML_DIR,
        max_crops_per_image=MAX_CROPS_PER_IMAGE,
    )

    crop_records_by_split[split_name] = records
    diagnostics_by_split[split_name] = diagnostics

    print(
        f"{split_name}: {len(limited_image_records[split_name]):,} "
        f"source images → {len(records):,} object crops"
    )
    print(
        f"  missing images={len(diagnostics['missing_images'])}, "
        f"missing XML={len(diagnostics['missing_xml'])}, "
        f"invalid XML={len(diagnostics['invalid_xml'])}, "
        f"XML with no valid boxes={len(diagnostics['empty_xml'])}, "
        f"multi-name XML={len(diagnostics['multi_name_images'])}"
    )

if any(not crop_records_by_split[name] for name in ("train", "val", "test")):
    raise ValueError(
        "At least one split produced no crops. "
        "Check the image/XML paths and filename matching."
    )

# Verify source-image separation after expanding to crops.
source_names = {
    split_name: {
        record["filename"]
        for record in crop_records_by_split[split_name]
    }
    for split_name in ("train", "val", "test")
}

assert source_names["train"].isdisjoint(source_names["val"])
assert source_names["train"].isdisjoint(source_names["test"])
assert source_names["val"].isdisjoint(source_names["test"])

manifest = {
    "configuration": {
        "box_padding": BOX_PADDING,
        "max_crops_per_image": MAX_CROPS_PER_IMAGE,
    },
    "splits": crop_records_by_split,
    "diagnostics": diagnostics_by_split,
}

with CROP_MANIFEST_PATH.open("w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

print(f"Saved crop manifest: {CROP_MANIFEST_PATH}")

ValueError: Invalid XML file: ..\pest-management-dataset\IP102_v1.1-20260731T031124Z-1-001\IP102_v1.1\Detection\VOC2007\Annotations\Annotations\IP087000986.xml

## 6. Inspect annotation assumptions

This section shows:

- object-name frequencies;
- how many crops each class receives;
- examples of XML files containing multiple distinct object names.

The XML object name is not used directly as the training target because the supplied top-15 JSON already contains remapped labels `0–14`, while no corresponding original-name-to-remapped-label table was supplied here.

In [ ]:
# 6. Diagnostics

for split_name in ("train", "val", "test"):
    object_name_counts = Counter(
        record["object_name"]
        for record in crop_records_by_split[split_name]
    )
    label_counts = Counter(
        record["label"]
        for record in crop_records_by_split[split_name]
    )

    print(f"\n{split_name.upper()}")
    print("Most common XML object names:")
    print(object_name_counts.most_common(10))
    print("Crop count by remapped label:")
    print(dict(sorted(label_counts.items())))
    print(
        "Invalid XML files:",
        len(diagnostics_by_split[split_name]["invalid_xml"]),
)

multi_name_examples = (
    diagnostics_by_split["train"]["multi_name_images"][:5]
)

if multi_name_examples:
    print("\nExample XML files with multiple distinct object names:")
    for example in multi_name_examples:
        print(example)
else:
    print("\nNo multi-name XML examples found in the selected training data.")

## 7. Crop-aware Dataset

The Dataset:

1. opens the original detection image;
2. retrieves the box belonging to one XML `<object>`;
3. validates and clips the coordinates;
4. adds optional padding;
5. crops the object;
6. resizes and transforms the crop;
7. returns the crop tensor and remapped class label.

In [ ]:
# 7. Transformations and crop Dataset

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15,
        hue=0.02,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])

evaluation_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])


class XmlCropDataset(Dataset):
    def __init__(
        self,
        image_dir,
        crop_records,
        transform=None,
        box_padding=0.05,
        min_crop_width=4,
        min_crop_height=4,
    ):
        self.image_dir = Path(image_dir)
        self.crop_records = crop_records
        self.transform = transform
        self.box_padding = float(box_padding)
        self.min_crop_width = int(min_crop_width)
        self.min_crop_height = int(min_crop_height)

        if not self.crop_records:
            raise ValueError("Dataset has no crop records.")

    def __len__(self):
        return len(self.crop_records)

    def _validated_crop_box(self, image, raw_box):
        left, top, right, bottom = map(float, raw_box)
        image_width, image_height = image.size

        box_width = right - left
        box_height = bottom - top

        if (
            box_width < self.min_crop_width
            or box_height < self.min_crop_height
        ):
            raise ValueError(
                f"Bounding box is too small: {raw_box}"
            )

        horizontal_padding = box_width * self.box_padding
        vertical_padding = box_height * self.box_padding

        left = max(
            0,
            int(round(left - horizontal_padding)),
        )
        top = max(
            0,
            int(round(top - vertical_padding)),
        )
        right = min(
            image_width,
            int(round(right + horizontal_padding)),
        )
        bottom = min(
            image_height,
            int(round(bottom + vertical_padding)),
        )

        if (
            right - left < self.min_crop_width
            or bottom - top < self.min_crop_height
        ):
            raise ValueError(
                f"Clipped bounding box is too small: "
                f"{(left, top, right, bottom)}"
            )

        return left, top, right, bottom

    def __getitem__(self, index):
        record = self.crop_records[index]
        image_path = self.image_dir / record["filename"]

        with Image.open(image_path) as image_file:
            image = image_file.convert("RGB")

        crop_box = self._validated_crop_box(
            image,
            record["box"],
        )
        crop = image.crop(crop_box)

        if self.transform is not None:
            crop = self.transform(crop)

        return crop, int(record["label"])

## 8. DataLoaders

Multiple crops from the same source image remain inside the same split because splitting happened at image level before crop expansion.

In [ ]:
# 8. Datasets and DataLoaders

train_dataset = XmlCropDataset(
    DETECTION_IMAGE_DIR,
    crop_records_by_split["train"],
    transform=train_transform,
    box_padding=BOX_PADDING,
    min_crop_width=MIN_CROP_WIDTH,
    min_crop_height=MIN_CROP_HEIGHT,
)
val_dataset = XmlCropDataset(
    DETECTION_IMAGE_DIR,
    crop_records_by_split["val"],
    transform=evaluation_transform,
    box_padding=BOX_PADDING,
    min_crop_width=MIN_CROP_WIDTH,
    min_crop_height=MIN_CROP_HEIGHT,
)
test_dataset = XmlCropDataset(
    DETECTION_IMAGE_DIR,
    crop_records_by_split["test"],
    transform=evaluation_transform,
    box_padding=BOX_PADDING,
    min_crop_width=MIN_CROP_WIDTH,
    min_crop_height=MIN_CROP_HEIGHT,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

images, labels = next(iter(train_loader))

print("Crop batch shape:", images.shape)
print("Label batch shape:", labels.shape)

## 9. Manual VGG-19

The architecture is kept equivalent to Notebook 1 so the primary experimental difference is the input:

- Notebook 1: complete classification images;
- Notebook 2: XML-localized object crops.

In [ ]:
# 9. VGG-19 model

class VGG19(nn.Module):
    def __init__(
        self,
        num_classes,
        strict_classifier=True,
        small_classifier_units=512,
    ):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 2
            nn.Conv2d(64, 128, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 3
            nn.Conv2d(128, 256, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 4
            nn.Conv2d(256, 512, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 5
            nn.Conv2d(512, 512, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )

        self.adaptive_pool = nn.AdaptiveAvgPool2d((7, 7))
        hidden_units = (
            4096 if strict_classifier else small_classifier_units
        )

        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, hidden_units),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(hidden_units, num_classes),
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for layer in self.modules():
            if isinstance(layer, nn.Conv2d):
                nn.init.kaiming_normal_(
                    layer.weight,
                    mode="fan_out",
                    nonlinearity="relu",
                )
                if layer.bias is not None:
                    nn.init.constant_(layer.bias, 0)

            elif isinstance(layer, nn.Linear):
                nn.init.normal_(layer.weight, 0, 0.01)
                nn.init.constant_(layer.bias, 0)

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)


model = VGG19(
    num_classes=NUM_CLASSES,
    strict_classifier=STRICT_VGG19_CLASSIFIER,
    small_classifier_units=SMALL_CLASSIFIER_UNITS,
).to(device)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"Trainable parameters: {parameter_count:,}")

with torch.no_grad():
    output = model(
        torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    )

print("Output shape:", output.shape)
assert output.shape == (1, NUM_CLASSES)

## 10. Loss, optimizer, scheduler, and epoch functions

In [ ]:
# 10. Training components and helper functions

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    progress = tqdm(loader, desc="Training", leave=False)

    for inputs, labels in progress:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        predictions = logits.argmax(dim=1)

        total_loss += loss.item() * batch_size
        total_correct += (predictions == labels).sum().item()
        total_samples += batch_size

        progress.set_postfix(
            loss=f"{total_loss / total_samples:.4f}",
            accuracy=f"{100 * total_correct / total_samples:.2f}%",
        )

    return (
        total_loss / total_samples,
        100 * total_correct / total_samples,
    )


def evaluate(model, loader, criterion, device, description):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        progress = tqdm(loader, desc=description, leave=False)

        for inputs, labels in progress:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(inputs)
            loss = criterion(logits, labels)

            batch_size = labels.size(0)
            predictions = logits.argmax(dim=1)

            total_loss += loss.item() * batch_size
            total_correct += (predictions == labels).sum().item()
            total_samples += batch_size

            progress.set_postfix(
                loss=f"{total_loss / total_samples:.4f}",
                accuracy=f"{100 * total_correct / total_samples:.2f}%",
            )

    return (
        total_loss / total_samples,
        100 * total_correct / total_samples,
    )

## 11. Train and save the best validation checkpoint

In [ ]:
# 11. Training loop

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
}

best_val_accuracy = float("-inf")

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")

    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
    )

    val_loss, val_accuracy = evaluate(
        model,
        val_loader,
        criterion,
        device,
        description="Validation",
    )

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(
        f"Train loss={train_loss:.4f}, "
        f"train accuracy={train_accuracy:.2f}%"
    )
    print(
        f"Val loss={val_loss:.4f}, "
        f"val accuracy={val_accuracy:.2f}%"
    )

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch + 1,
                "best_val_accuracy": best_val_accuracy,
                "config": {
                    "image_size": IMAGE_SIZE,
                    "num_classes": NUM_CLASSES,
                    "box_padding": BOX_PADDING,
                },
            },
            BEST_MODEL_PATH,
        )

        print(f"Saved best checkpoint: {BEST_MODEL_PATH}")

## 12. Learning curves and final test evaluation

In [ ]:
# 12. Plot learning curves

epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_accuracy"], label="Training accuracy")
plt.plot(epochs, history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("XML-Crop Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_loss"], label="Training loss")
plt.plot(epochs, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("XML-Crop Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 13. Restore the best model and test once

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_accuracy = evaluate(
    model,
    test_loader,
    criterion,
    device,
    description="Testing",
)

print(f"Best checkpoint epoch: {checkpoint['epoch']}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2f}%")

## Comparing Notebook 1 and Notebook 2

A fair comparison should keep these settings equal where practical:

- VGG-19 architecture;
- image size;
- optimizer;
- learning rate;
- epoch count;
- augmentation strength;
- checkpoint-selection rule.

However, the two datasets differ naturally:

- Notebook 1 uses 102 whole-image classes unless you deliberately filter it;
- Notebook 2 uses the 15 remapped classes provided by `splits_top15.json`;
- Notebook 2 may produce more than one crop per source image.

Therefore, their raw accuracies are not a perfectly controlled “cropping only” comparison unless Notebook 1 is also filtered to the same 15 classes and equivalent source population. They are still useful as two distinct experiments: broad whole-image classification versus localized top-15 object classification.